In [ ]:
%load_ext autoreload
%autoreload 2

In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import glob
import os
from pathlib import Path
from tqdm import tqdm
import gzip
import holoviews as hv
import hdf5storage
import hashlib

import unfoc

from opr_ingest.utig import file_index, stream_util, segment_splits, transects, postprocessed_gps, gps_pipeline, headers
from opr_ingest.core import basemap, geo, opr_gps_matlab, opr_headers, params

In [ ]:
pd.options.mode.copy_on_write = True
tqdm.pandas()
hv.extension('bokeh')

In [ ]:
use_cache = True
cache_dir = "../../outputs/file_index.csv"
base_path = "/kucresis/scratch/data/UTIG"

df_files = file_index.load_file_index_df(base_path, cache_dir, read_cache=use_cache)
df_artifacts = file_index.create_artifacts_df(df_files) # df_artifacts is a dataframe with one row per stream file

df_artifacts.head()

In [ ]:
# Find all transects matching a specific prj and trn
prj, trn = 'ASB', 'R04Ea'
df_tmp = df_artifacts[(df_artifacts['prj'] == prj) & (df_artifacts['trn'] == trn)]

file_results = {}

for idx, row in df_tmp.iterrows():
    if ('GPS' in row['stream']) or ('RAD' in row['stream']):
        # Calculate MD5 checksum
        chk = hashlib.md5(open(row['full_path'],'rb').read()).hexdigest()

        print(f"Stream type: {row['stream']}, Path: {row['full_path']}, MD5 (last 4): {chk[-4:]}")

        # Load file
        if 'GPS' in row['stream']:
            df = stream_util.load_xds_stream_file(row['full_path'])
        else:
            df = stream_util.load_ct_file(row['full_path'])

        file_results[row['full_path']] = {
            'md5': chk,
            'df': df,
            'stream': row['stream']
        }